# 3. Outlier Analysis
Investigate unusual annual spending before clustering. Flagging is not proof of error.
All customers remain available for final assignment. This notebook does not refit preprocessing or choose the final clustering representation.

## 1. Imports and data
Run from the repository root or its `notebooks` folder. Dependencies: numpy, pandas, plotly, scikit-learn, minisom. Install these in your notebook kernel environment if needed.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from utils.outlier_functions_check import (
    calculate_iqr_bounds, flag_iqr_outliers, plot_spending_distribution,
    calculate_neighbor_distances, evaluate_dbscan, detect_kmeans_outliers,
    detect_som_outliers, summarize_flags, plot_outlier_map,
)

In [2]:
data_path = project_root / "data" / "preprocessed" / "customer_info_clean.csv"
customer_info= pd.read_csv(data_path)
print(f"Customers: {len(customer_info):,}; features: {customer_info.shape[1]}")
customer_info.head()

Customers: 34,060; features: 25


,customer_id,distinct_stores_visited,lifetime_total_distinct_products,pct_prod_in_promotion,year_first_transaction,latitude,longitude,age,degree,loyalty_card,...,spend_per_year_groceries,spend_per_year_electronics,spend_per_year_vegetables,spend_per_year_nonalcohol_drinks,spend_per_year_alcohol_drinks,spend_per_year_meat,spend_per_year_fish,spend_per_year_hygiene,spend_per_year_petfood,spend_per_year_videogames
0,29930,4,386.0,0.158741,2018.0,38.721807,-9.125534,53,0,0,...,1112.714286,800.142857,103.714286,137.428571,173.285714,228.285714,270.571429,65.285714,61.142857,58.857143
1,6813,4,73.0,0.122789,2013.0,38.734668,-9.163533,81,1,1,...,721.083333,2.916667,66.000000,8.500000,8.666667,61.750000,28.833333,32.833333,18.833333,6.250000
2,39451,7,319.0,0.101598,2011.0,38.787126,-9.147077,36,0,0,...,1114.642857,305.357143,113.214286,70.000000,133.714286,94.500000,140.785714,65.714286,13.714286,23.928571
3,21557,1,221.0,0.259943,2009.0,38.741816,-9.159700,49,0,0,...,840.000000,1022.875000,1.750000,16.812500,115.937500,58.687500,49.062500,8.687500,16.875000,42.437500
4,16415,5,244.0,0.317822,2012.0,38.785921,-9.149221,59,1,1,...,3788.461538,245.923077,19.846154,55.846154,42.076923,75.615385,114.769231,80.461538,11.076923,8.615385


## 2. Spending feature selection
Use only the ten annual spending categories. Exclude IDs, totals, location, categorical indicators and demographics from detector distances; keep them for profiling. Screening uses original monetary values for IQR and robust-scaled values for multidimensional detectors.

In [3]:
spend_columns = [c for c in customer_info.columns if c.startswith("spend_per_year_")]
X = customer_info[spend_columns].copy()
assert len(spend_columns) == 10
assert customer_info["customer_id"].notna().all() and customer_info["customer_id"].is_unique
assert X.notna().all().all()
assert np.isfinite(X.to_numpy()).all()
assert X.ge(0).all().all()
X.describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).T.round(2)

,count,mean,std,min,25%,50%,75%,95%,99%,max
spend_per_year_groceries,34060.0,2927.74,4196.25,0.00,941.98,1763.54,3328.31,9226.90,19330.23,88706.0
spend_per_year_electronics,34060.0,518.65,854.51,0.00,99.62,267.94,544.39,1981.02,3983.73,26214.0
spend_per_year_vegetables,34060.0,96.53,149.79,0.00,22.40,49.62,122.88,287.67,768.81,2602.0
spend_per_year_nonalcohol_drinks,34060.0,60.31,71.62,0.00,26.64,43.25,68.00,164.00,374.61,1547.0
spend_per_year_alcohol_drinks,34060.0,110.73,153.38,0.06,33.93,70.44,133.17,318.20,784.24,2908.0
spend_per_year_meat,34060.0,117.12,134.11,0.00,54.80,89.12,139.14,302.25,720.70,2208.0
spend_per_year_fish,34060.0,105.43,142.89,0.00,28.79,72.32,129.00,310.75,684.70,2886.0
spend_per_year_hygiene,34060.0,89.38,115.67,0.00,34.82,63.50,103.83,242.50,550.82,2734.0
spend_per_year_petfood,34060.0,41.70,45.64,0.00,19.42,30.00,47.50,110.75,239.50,796.0
spend_per_year_videogames,34060.0,54.34,72.85,0.00,17.17,29.50,59.44,191.01,342.60,1872.0


## 3. Univariate analysis
10×IQR is the previous conservative baseline, not a validated removal rule. Negative lower bounds are retained in the table; non-negative spending cannot breach those bounds. Standard box-plot whiskers differ from the custom histogram screening lines.

In [4]:
iqr_multiplier = 10
bounds = calculate_iqr_bounds(customer_info, spend_columns, iqr_multiplier)
iqr_flags = flag_iqr_outliers(customer_info, bounds)
iqr_summary = bounds.copy()
iqr_summary["customers_flagged"] = iqr_flags.sum()
iqr_summary["percentage_flagged"] = iqr_flags.mean() * 100
outlier_flags = customer_info[["customer_id"]].copy()
outlier_flags["iqr_outlier"] = iqr_flags.any(axis=1)
outlier_flags["iqr_features_flagged"] = iqr_flags.sum(axis=1)
iqr_summary.sort_values("customers_flagged", ascending=False).round(2)

,q1,q3,iqr,lower_bound,upper_bound,customers_flagged,percentage_flagged
spend_per_year_nonalcohol_drinks,26.64,68.00,41.36,-387.00,481.64,188,0.55
spend_per_year_electronics,99.62,544.39,444.77,-4348.12,4992.13,170,0.50
spend_per_year_alcohol_drinks,33.93,133.17,99.24,-958.47,1125.56,167,0.49
spend_per_year_hygiene,34.82,103.83,69.02,-655.33,793.98,163,0.48
spend_per_year_groceries,941.98,3328.31,2386.34,-22921.38,27191.66,160,0.47
spend_per_year_petfood,19.42,47.50,28.08,-261.37,328.29,159,0.47
spend_per_year_meat,54.80,139.14,84.34,-788.56,982.49,142,0.42
spend_per_year_videogames,17.17,59.44,42.28,-405.61,482.22,105,0.31
spend_per_year_fish,28.79,129.00,100.21,-973.28,1131.08,104,0.31
spend_per_year_vegetables,22.40,122.88,100.48,-982.37,1127.65,101,0.30


In [5]:
sensitivity = []
for multiplier in [3, 5, 10]:
    flags = flag_iqr_outliers(customer_info, calculate_iqr_bounds(customer_info, spend_columns, multiplier))
    selected = flags.any(axis=1)
    sensitivity.append({"iqr_multiplier": multiplier, "customers_flagged": selected.sum(),
                        "percentage_flagged": selected.mean() * 100})
sensitivity = pd.DataFrame(sensitivity)
sensitivity.round(2)

,iqr_multiplier,customers_flagged,percentage_flagged
0,3,4420,12.98
1,5,2160,6.34
2,10,614,1.80


In [6]:
# Change the category here to inspect one readable chart at a time
plot_spending_distribution(customer_info, "spend_per_year_groceries", bounds).show()

## 4. Tenure diagnostics before model fitting
Zero tenure is not automatically a data error. Under the current upstream policy its annual spending is estimated rather than divided by a positive tenure. This table does not reconstruct missing-value provenance: retain original imputation flags upstream in a future revision.

Review zero- and one-year customers separately. Do not silently change tenure to one or shift the reference year to remove flags.

In [7]:
review = customer_info.merge(outlier_flags, on="customer_id", validate="one_to_one")
tenure_summary = review.groupby("customer_lifetime").agg(
    customers=("customer_id", "size"), customers_flagged=("iqr_outlier", "sum"),
    flag_rate=("iqr_outlier", "mean"))
tenure_summary["flag_rate"] *= 100
display(tenure_summary.round(2))
display(review.loc[review["iqr_outlier"],
    ["customer_id", "age", "customer_lifetime", "iqr_features_flagged"] + spend_columns]
    .sort_values("iqr_features_flagged", ascending=False).head(15))

,customers,customers_flagged,flag_rate
customer_lifetime,,,
0.0,345,89,25.80
1.0,310,286,92.26
2.0,401,145,36.16
3.0,365,27,7.40
4.0,3163,62,1.96
5.0,1457,5,0.34
6.0,1738,0,0.00
7.0,2060,0,0.00
8.0,2369,0,0.00


,customer_id,age,customer_lifetime,iqr_features_flagged,spend_per_year_groceries,spend_per_year_electronics,spend_per_year_vegetables,spend_per_year_nonalcohol_drinks,spend_per_year_alcohol_drinks,spend_per_year_meat,spend_per_year_fish,spend_per_year_hygiene,spend_per_year_petfood,spend_per_year_videogames
15898,8643,25,1.0,10,45292.0,5821.0,1608.000000,690.0,1704.0,1255.0,1403.0,1405.0,393.0,527.0
21494,34287,25,1.0,9,60693.0,9201.0,1491.000000,545.0,1669.0,1433.0,2358.0,1337.0,116.0,625.0
16704,39014,25,1.0,9,72821.0,1799.0,2213.000000,520.0,1543.0,1845.0,1755.0,913.0,546.0,703.0
17777,22805,25,1.0,9,68766.0,2818.0,2499.000000,490.0,1576.0,1157.0,1538.0,1144.0,437.0,660.0
21467,5715,25,1.0,9,80809.0,8271.0,1658.000000,761.0,1772.0,1011.0,2724.0,800.0,263.0,501.0
24682,2939,25,1.0,9,66933.0,5594.0,2120.000000,555.0,1682.0,1563.0,928.0,1103.0,553.0,631.0
28295,24630,25,1.0,8,37433.0,5371.0,908.000000,414.0,1385.0,1335.0,1373.0,1052.0,577.0,513.0
12363,12680,25,1.0,8,77707.0,4479.0,2166.000000,505.0,1429.0,1176.0,1774.0,1449.0,169.0,498.0
14272,20985,25,1.0,8,10946.0,6765.0,610.000000,623.0,1732.0,1379.0,1240.0,1205.0,420.0,851.0
21456,11399,25,1.0,8,60890.0,10424.0,711.000000,508.0,1346.0,947.0,1979.0,1175.0,361.0,565.0


### Detector fitting population
Default: fit multidimensional detectors on positive-tenure customers only, without deleting anyone from the customer table. Zero/negative-tenure rows are marked not evaluated by those detectors, not as confirmed inliers. This is a temporary analytical separation while the annualization policy is reviewed. All positive-tenure customers, including IQR flags, enter all detectors.

You may include zero-tenure customers only after reviewing their upstream estimates. RobustScaler is fitted on this selected population; it changes scale, not skew. Notebook 4 will compare final feature representations.

In [8]:
include_zero_tenure = False
eligible = customer_info["customer_lifetime"].gt(0)
if include_zero_tenure:
    eligible = customer_info["customer_lifetime"].ge(0)

outlier_flags["multidimensional_evaluated"] = eligible
outlier_flags["tenure_review_required"] = customer_info["customer_lifetime"].le(0)
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X.loc[eligible])
print(f"Evaluated: {eligible.sum():,}; not evaluated: {(~eligible).sum():,}")

Evaluated: 33,715; not evaluated: 345


## 5. Multidimensional analysis
### 5.1 DBSCAN
Inspect neighbour distances and candidate noise rates. Candidate eps values below are exploratory values derived from distance quantiles, not an optimal parameter search. DBSCAN may use substantial memory for large eps. Start with this small candidate set; do not force a chosen noise percentage.

In [9]:
min_samples = 10
neighbor_distances = calculate_neighbor_distances(X_scaled, min_samples)
px.line(x=np.arange(len(neighbor_distances)), y=neighbor_distances,
        labels={"x": "Customers sorted by neighbour distance", "y": "Neighbour distance"},
        title="DBSCAN neighbourhood diagnostic", template="plotly_white").show()

In [10]:
eps_values = np.unique(np.quantile(neighbor_distances, [0.80, 0.90, 0.95]))
eps_values = eps_values[eps_values > 0]
dbscan_results = evaluate_dbscan(X_scaled, eps_values, min_samples)
dbscan_results

,eps,min_samples,clusters,customers_flagged,percentage_flagged
0,1.358177,10,2,5194,15.405606
1,2.092986,10,1,2630,7.800682
2,3.086028,10,1,1143,3.390182


In [11]:
# Provisional starting candidate: review the table and distance curve before accepting
chosen_eps = float(eps_values[len(eps_values) // 2])
dbscan_labels = DBSCAN(eps=chosen_eps, min_samples=min_samples).fit_predict(X_scaled)
outlier_flags["dbscan_outlier"] = pd.Series(pd.NA, index=customer_info.index, dtype="boolean")
outlier_flags.loc[eligible, "dbscan_outlier"] = dbscan_labels == -1
print(f"Provisional eps: {chosen_eps:.4f}")

Provisional eps: 2.0930


### 5.2 High-K K-Means
70 prototypes and the 97th-percentile distance are starting settings carried over from the original approach. The cutoff flags roughly the highest-distance 3% by construction; that is not evidence of a true 3% error rate.

In [12]:
kmeans_scores, kmeans_flags, kmeans_threshold, kmeans = detect_kmeans_outliers(
    X_scaled, n_clusters=70, percentile=97)
outlier_flags["kmeans_outlier"] = pd.Series(pd.NA, index=customer_info.index, dtype="boolean")
outlier_flags["kmeans_distance"] = np.nan
outlier_flags.loc[eligible, "kmeans_outlier"] = kmeans_flags
outlier_flags.loc[eligible, "kmeans_distance"] = kmeans_scores

### 5.3 SOM
A 10×10 map with 10,000 random updates is exploratory, not assumed converged. Inspect score/flag sensitivity to training length and random seed before finalizing treatment. This section requires minisom.

In [13]:
som_scores, som_flags, som_threshold, som = detect_som_outliers(
    X_scaled, size=10, iterations=10000, percentile=97)
outlier_flags["som_outlier"] = pd.Series(pd.NA, index=customer_info.index, dtype="boolean")
outlier_flags["som_distance"] = np.nan
outlier_flags.loc[eligible, "som_outlier"] = som_flags
outlier_flags.loc[eligible, "som_distance"] = som_scores

## 6. Agreement and sensitivity
Agreement is corroboration, not independent proof: the methods share the same inputs and two use prototype distances. Missing detector flags mean not evaluated. Percentages below use the evaluated population only.

In [14]:
method_columns = ["dbscan_outlier", "kmeans_outlier", "som_outlier"]
evaluated_flags = outlier_flags.loc[eligible, method_columns].astype(bool)
method_summary = summarize_flags(evaluated_flags, method_columns)
display(method_summary.round(2))
outlier_flags["consensus_outlier"] = pd.Series(pd.NA, index=customer_info.index, dtype="boolean")
outlier_flags.loc[eligible, "consensus_outlier"] = evaluated_flags.all(axis=1)
agreement = evaluated_flags.value_counts().rename("customers").reset_index()
display(agreement)
print("IQR flags across all customers:", int(outlier_flags["iqr_outlier"].sum()))

,customers_flagged,percentage_flagged
dbscan_outlier,2630,7.8
kmeans_outlier,1012,3.0
som_outlier,1012,3.0


,dbscan_outlier,kmeans_outlier,som_outlier,customers
0,False,False,False,31085
1,True,False,False,1479
2,True,True,True,873
3,True,True,False,139
4,True,False,True,139


IQR flags across all customers: 614


In [15]:
score_sensitivity = []
for method, scores in [("K-Means", kmeans_scores), ("SOM", som_scores)]:
    for percentile in [95, 97, 99]:
        flags = scores > np.percentile(scores, percentile)
        score_sensitivity.append({"method": method, "percentile": percentile,
                                  "customers_flagged": flags.sum(), "percentage_flagged": flags.mean() * 100})
pd.DataFrame(score_sensitivity).round(2)

,method,percentile,customers_flagged,percentage_flagged
0,K-Means,95,1686,5.0
1,K-Means,97,1012,3.0
2,K-Means,99,338,1.0
3,SOM,95,1686,5.0
4,SOM,97,1012,3.0
5,SOM,99,338,1.0


### Shared diagnostic map
Fit one two-component PCA projection and reuse it across methods. This is visualization only, not the final dimensionality-reduction choice. Edge position and visible separation do not validate removal; return to original-unit profiles.

In [16]:
diagnostic_pca = PCA(n_components=2)
embedding = diagnostic_pca.fit_transform(X_scaled)
print("Variance shown:", diagnostic_pca.explained_variance_ratio_.sum().round(3))
for column in method_columns:
    plot_outlier_map(embedding, evaluated_flags[column], column.replace("_", " ").title()).show()

Variance shown: 0.71


## 7. Review flagged profiles
Compare medians in original spending units. Inspect individual customers and tenure, and consult raw records where a data error is suspected. A coherent high-spending group may be a valuable segment rather than disposable noise.

In [17]:
review = customer_info.merge(outlier_flags, on="customer_id", validate="one_to_one")
evaluated_review = review.loc[review["multidimensional_evaluated"]]
display(evaluated_review.groupby("consensus_outlier")[spend_columns].median().T.round(2))
display(evaluated_review.loc[evaluated_review["consensus_outlier"].fillna(False),
    ["customer_id", "age", "customer_lifetime", "iqr_features_flagged"] + spend_columns].head(20))

consensus_outlier,False,True
spend_per_year_groceries,1695.44,11736.33
spend_per_year_electronics,256.60,2133.50
spend_per_year_vegetables,47.22,237.00
spend_per_year_nonalcohol_drinks,42.07,258.50
spend_per_year_alcohol_drinks,67.11,488.67
spend_per_year_meat,86.93,451.00
spend_per_year_fish,69.56,457.00
spend_per_year_hygiene,61.61,361.67
spend_per_year_petfood,29.25,169.00
spend_per_year_videogames,28.40,210.50


,customer_id,age,customer_lifetime,iqr_features_flagged,spend_per_year_groceries,spend_per_year_electronics,spend_per_year_vegetables,spend_per_year_nonalcohol_drinks,spend_per_year_alcohol_drinks,spend_per_year_meat,spend_per_year_fish,spend_per_year_hygiene,spend_per_year_petfood,spend_per_year_videogames
74,296,26,2.0,1,11099.000000,7997.500000,213.000000,293.500000,139.000000,549.500000,379.050000,361.000000,208.500000,137.000000
142,10996,26,2.0,0,10726.000000,1214.500000,948.000000,233.000000,302.000000,24.500000,13.000000,676.500000,77.000000,138.000000
226,18622,25,1.0,1,9861.000000,775.000000,223.000000,173.000000,1453.000000,298.000000,923.000000,259.000000,172.000000,273.000000
258,18824,26,2.0,2,43635.500000,2111.500000,968.000000,216.000000,673.000000,730.000000,1354.000000,337.000000,179.500000,282.000000
373,34164,25,1.0,5,45490.000000,3354.000000,51.000000,1547.000000,960.000000,1442.000000,1030.800000,960.000000,518.000000,276.000000
393,15709,27,3.0,0,19283.333333,2210.000000,842.666667,146.666667,360.333333,464.000000,694.000000,326.333333,179.333333,174.000000
401,30083,26,2.0,1,4991.500000,4573.000000,177.000000,135.500000,335.000000,756.000000,592.500000,804.500000,56.500000,160.500000
414,11038,26,2.0,0,9604.000000,1032.500000,145.000000,356.000000,510.500000,617.500000,212.000000,757.000000,183.500000,157.500000
417,39275,26,2.0,0,13572.000000,3018.000000,127.000000,257.500000,286.500000,516.000000,219.500000,581.000000,139.000000,105.500000
422,9985,78,5.0,0,3126.200000,526.000000,234.600000,108.600000,686.200000,203.200000,149.800000,58.400000,114.200000,49.600000


In [18]:
consensus_tenure = (
    evaluated_review
    .groupby("customer_lifetime")
    .agg(
        customers=("customer_id", "size"),
        consensus_flagged=("consensus_outlier", "sum"),
        flag_rate=("consensus_outlier", "mean"),
    )
)

consensus_tenure["flag_rate"] *= 100

consensus_tenure.round(2)

,customers,consensus_flagged,flag_rate
customer_lifetime,,,
1.0,310,302,97.42
2.0,401,278,69.33
3.0,365,123,33.7
4.0,3163,157,4.96
5.0,1457,9,0.62
6.0,1738,2,0.12
7.0,2060,2,0.1
8.0,2369,0,0.0
9.0,2566,0,0.0


In [19]:
pd.crosstab(
    evaluated_review["iqr_outlier"],
    evaluated_review["consensus_outlier"],
    rownames=["IQR flag"],
    colnames=["Consensus flag"],
)

Consensus flag,False,True
IQR flag,,
False,32780,410
True,62,463


## 8. Treatment decision and export
No customer is excluded automatically. Enter reviewed customer IDs and reasons below only after inspection. The default exports the full customer dataset as the fitting candidate, with zero-tenure cases explicitly unresolved. Notebook 4 must resolve those cases before final model fitting.

Finalizing this notebook requires recording: chosen DBSCAN settings and rationale; threshold sensitivity; whether flags reflect plausible customers or errors; the tenure policy; and any exclusion/reassignment plan. Every customer must ultimately receive a segment.

In [20]:
# Leave empty until the flagged records have been reviewed
reviewed_exclusions = {
    # customer_id: "Documented reason for exclusion from fitting",
}

outlier_flags["exclude_from_fitting"] = outlier_flags["customer_id"].isin(reviewed_exclusions)
outlier_flags["exclusion_reason"] = outlier_flags["customer_id"].map(reviewed_exclusions).fillna("")
customer_info_for_features = customer_info.loc[~outlier_flags["exclude_from_fitting"]].copy()
customers_excluded = customer_info.loc[outlier_flags["exclude_from_fitting"]].copy()

assert len(customer_info_for_features) + len(customers_excluded) == len(customer_info)
print(f"Unresolved tenure cases: {outlier_flags['tenure_review_required'].sum():,}")
print(f"Reviewed exclusions: {len(customers_excluded):,}")

Unresolved tenure cases: 345
Reviewed exclusions: 0


In [21]:
tables_dir = project_root / "reports" / "tables"
data_dir = project_root / "data" / "preprocessed"
tables_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

outlier_flags.to_csv(data_dir / "customer_outlier_flags.csv", index=False)
customer_info_for_features.to_csv(data_dir / "customer_info_for_features.csv", index=False)
customers_excluded.to_csv(data_dir / "customers_excluded_from_fitting.csv", index=False)
iqr_summary.to_csv(tables_dir / "iqr_summary.csv")
sensitivity.to_csv(tables_dir / "iqr_sensitivity.csv", index=False)
tenure_summary.to_csv(tables_dir / "outlier_tenure_summary.csv")
dbscan_results.to_csv(tables_dir / "dbscan_candidates.csv", index=False)
method_summary.to_csv(tables_dir / "outlier_method_summary.csv")
agreement.to_csv(tables_dir / "outlier_agreement.csv", index=False)

settings = pd.Series({"iqr_multiplier": iqr_multiplier, "include_zero_tenure": include_zero_tenure,
    "scaling": "RobustScaler on eligible original-unit spending", "dbscan_eps": chosen_eps,
    "dbscan_min_samples": min_samples, "kmeans_clusters": 70, "distance_percentile": 97,
    "som_size": "10x10", "som_iterations": 10000, "random_seed": 42})
settings.to_csv(tables_dir / "outlier_settings.csv", header=["value"])
print("Exports complete. Keep flags separate from the modeling feature matrix.")

Exports complete. Keep flags separate from the modeling feature matrix.


## Handoff to 4_Feature_Preparation.ipynb

## Conclusion and Handoff

Outlier flags are retained as diagnostic information rather than used as
automatic removal rules.

All positive-tenure customers remain eligible for the baseline clustering
analysis, including customers flagged by the screening methods.

Zero-tenure customers are preserved but remain outside baseline model
fitting until their annual-spending estimates have been reviewed.

Notebook 4 will compare feature representations. Detector settings remain
exploratory; no claim is made that consensus flags identify confirmed errors.
Load `customer_info_for_features.csv` and join `customer_outlier_flags.csv` by customer_id for diagnostics, not as model features. Resolve tenure-review cases before fitting. Keep excluded customers and their IDs for a later, explicitly documented assignment strategy. Compare representations with and without reviewed exclusions rather than assuming removal improves segmentation.